# Virtual Experiment Executor Quickstart

A short walkthrough for generating CoAX-style simulated trial rows through `src.virtual_experiment_executor`.

In [2]:
from pathlib import Path
import sys

def find_repo_root(start=None):
    start = Path(start) if start else Path.cwd()
    for p in [start] + list(start.parents):
        if (p / "src").is_dir() or (p / "pyproject.toml").exists() or (p / "README.md").exists():
            return p
    return Path.cwd()

repo_root = find_repo_root()
sys.path.insert(0, str(repo_root))

import pandas as pd

from src.virtual_experiment_executor import (
    VirtualExperimentConfig,
    simulate_virtual_experiment,
    save_virtual_experiment_results,
)


## Generate Trial Rows

`max_participants=5` creates simulated participant IDs `1` through `5`. The executor uses the fitted cognitive parameter CSV internally, but the output does not expose the original participant IDs.

In [3]:
config = VirtualExperimentConfig(
    app_id="adult",
    xai_type="Importance",
    tested_with_xai="w/ XAI",
    max_participants=5,
    seed=7,
)

df = simulate_virtual_experiment(config)
df.shape


(456, 25)

In [4]:
df.head()


,Participant Id,appId,XAIType,Tested w/ XAI,Selected Strategy,Selected Strategy NLL,Strategy,Strategy NLL,Parameter Row Index,Parameter NLL,...,scaling_factor,explanation_type,Instance Index,trialType,predicted,Prob correct,Prob 0,Prob 1,Correct,response_time
0,1,adult,Importance,w/ XAI,,NaN,Sensitive-features categorization,4.160694,0,0.251206,...,NaN,NaN,260,Train,1,0.500000,5.000000e-01,0.500000,1,0.1
1,1,adult,Importance,w/ XAI,,NaN,Sensitive-features categorization,4.160694,0,0.251206,...,NaN,NaN,260,Train,1,0.500000,5.000000e-01,0.500000,1,0.1
2,1,adult,Importance,w/ XAI,,NaN,Sensitive-features categorization,4.160694,0,0.251206,...,NaN,NaN,141,Train,1,0.000000,0.000000e+00,1.000000,0,0.1
3,1,adult,Importance,w/ XAI,,NaN,Sensitive-features categorization,4.160694,0,0.251206,...,NaN,NaN,141,Train,1,0.000000,0.000000e+00,1.000000,0,0.1
4,1,adult,Importance,w/ XAI,,NaN,Sensitive-features categorization,4.160694,0,0.251206,...,NaN,NaN,17,Train,1,0.999999,5.476653e-07,0.999999,1,0.1


## Check Participants And Parameters

Each trial row includes the cognitive parameters used for that simulated participant-strategy record.

In [5]:
df["Participant Id"].unique().tolist()


[1, 2, 3, 4, 5]

In [6]:
parameter_cols = [
    "Participant Id",
    "Strategy",
    "Parameter Row Index",
    "Parameter NLL",
    "Parameter Session",
    "k",
    "decay_param",
    "sensitivity",
    "retrieval_threshold",
    "scaling_factor",
    "explanation_type",
]

df[parameter_cols].drop_duplicates().head(10)


,Participant Id,Strategy,Parameter Row Index,Parameter NLL,Parameter Session,k,decay_param,sensitivity,retrieval_threshold,scaling_factor,explanation_type
0,1,Sensitive-features categorization,0,0.251206,2,1,0.5,76.703639,-2.967453,NaN,NaN
76,2,Sensitive-features categorization,1,0.463745,2,3,0.5,15.440995,-1.423553,NaN,NaN
152,3,Sensitive-features categorization,2,0.401954,2,3,0.5,85.585153,-2.322480,NaN,NaN
228,3,Attribution Sum,214,0.625459,1,1,0.5,NaN,-3.841924,1.096794,importance
304,4,Sensitive-features categorization,3,0.270358,2,5,0.5,100.000000,-4.000000,NaN,NaN
380,5,Sensitive-features categorization,4,0.078042,2,5,0.5,75.860345,-1.976620,NaN,NaN


## Optional Best-Strategy Mode

By default, the executor keeps all available strategy parameter rows for each simulated participant. Use `select_best=True` only when you want to keep the lowest simulated-NLL strategy per participant condition.

In [7]:
best_config = VirtualExperimentConfig(
    app_id="adult",
    xai_type="Importance",
    tested_with_xai="w/ XAI",
    max_participants=5,
    seed=7,
    select_best=True,
)

best_df = simulate_virtual_experiment(best_config)
best_df[["Participant Id", "Selected Strategy", "Selected Strategy NLL"]].drop_duplicates()


,Participant Id,Selected Strategy,Selected Strategy NLL
0,1,Sensitive-features categorization,4.160765
76,2,Sensitive-features categorization,2.955263
152,3,Attribution Sum,0.812440
228,4,Sensitive-features categorization,5.249904
304,5,Sensitive-features categorization,5.121254


## Save Results

The default CLI writes to `simulated_results/coax_api_simulated_trials.csv`; the API can save to the same place.

In [8]:
output_path = repo_root / "simulated_results" / "coax_api_simulated_trials_from_notebook.csv"
save_virtual_experiment_results(df, output_path)


PosixPath('/Users/wangzhuoyulucas/Documents/GitHub/xaik-tool-cognitive-agent/simulated_results/coax_api_simulated_trials_from_notebook.csv')

## Equivalent CLI

The root script is a thin wrapper around the same API.

In [9]:
!python ../simulate_coax_trials_from_api.py --app-id adult --xai-type Importance --tested-with-xai "w/ XAI" --max-participants 5


Wrote 456 rows to /Users/wangzhuoyulucas/Documents/GitHub/xaik-tool-cognitive-agent/notebooks/simulated_results/coax_api_simulated_trials.csv
